In [7]:
import pandas as pd
import pandasql as ps 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
import warnings 
warnings.filterwarnings('ignore') 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries loaded successfully")

Libraries loaded successfully


In [ ]:
# Load All Datasets
import os
base_path = os.path.expanduser("~/Documents/Ecommers Project/Brazilian E-Commerce Dataset/")

# Load all 9 CSV files
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(base_path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(base_path + "product_category_name_translation.csv")

print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Order Payments: {order_payments.shape}")
print(f"Order Reviews: {order_reviews.shape}")
print(f"Products: {products.shape}")
print(f"Sellers: {sellers.shape}")
print(f"Geolocation: {geolocation.shape}")
print(f"Category Translation: {category_translation.shape}")
print("\nAll datasets loaded successfully!")

Orders: (99441, 8)
Customers: (99441, 5)
Order Items: (112650, 7)
Order Payments: (103886, 5)
Order Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)

All datasets loaded successfully!


In [9]:
# Convert Date Columns & Basic Overview
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
orders['order_year'] = orders['order_purchase_timestamp'].dt.year

print("Date columns converted")
print(f"\nDate range: {orders['order_purchase_timestamp'].min()} to {orders['order_purchase_timestamp'].max()}")
print(f"Total orders: {len(orders):,}")
print(f"\nOrder statuses:")
print(orders['order_status'].value_counts())

Date columns converted

Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
Total orders: 99,441

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [ ]:
# Monthly Revenue Analysis
# Merge orders with payments
orders_payments = orders.merge(order_payments, on='order_id', how='inner')
delivered = orders_payments[orders_payments['order_status'] == 'delivered'].copy()

# Drop Period columns — pandasql doesn't support them
delivered = delivered.drop(columns=['order_month', 'order_year'], errors='ignore')

# Convert all datetime columns to string
for col in delivered.select_dtypes(include=['datetime64[ns]']).columns:
    delivered[col] = delivered[col].astype(str)

query1 = """
SELECT 
    substr(order_purchase_timestamp, 1, 7) AS month,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    ROUND(AVG(payment_value), 2) AS avg_order_value
FROM delivered
GROUP BY month
ORDER BY month
"""

monthly_revenue = ps.sqldf(query1, locals())
print("=== MONTHLY REVENUE ===")
print(monthly_revenue.to_string())
print(f"\nTotal Revenue: BRL {order_payments['payment_value'].sum():,.2f}")
print(f"Total Delivered Orders: {delivered['order_id'].nunique():,}")
print(f"Average Order Value: BRL {order_payments['payment_value'].mean():,.2f}")

=== MONTHLY REVENUE ===
      month  total_orders  total_revenue  avg_order_value
0   2016-10           265       46566.71           165.13
1   2016-12             1          19.62            19.62
2   2017-01           750      127545.67           159.63
3   2017-02          1653      271298.65           155.12
4   2017-03          2546      414369.39           153.47
5   2017-04          2303      390952.18           160.49
6   2017-05          3546      567066.73           149.74
7   2017-06          3135      490225.60           147.53
8   2017-07          3872      566403.93           136.58
9   2017-08          4193      646000.61           147.05
10  2017-09          4150      701169.99           160.41
11  2017-10          4478      751140.27           159.89
12  2017-11          7289     1153528.05           151.92
13  2017-12          5513      843199.17           147.18
14  2018-01          7069     1078606.86           146.73
15  2018-02          6555      966510.88        

In [20]:
# Interactive Dashboard using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Monthly Revenue Trend (BRL)',
        'Monthly Order Volume',
        'Average Order Value Over Time',
        'Revenue Growth Rate (%)'
    ),
    specs=[[{"type": "bar"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# Calculate growth rate
monthly_revenue['revenue_growth'] = monthly_revenue['total_revenue'].pct_change() * 100

# Chart 1 — Monthly Revenue Bar
fig.add_trace(
    go.Bar(
        x=monthly_revenue['month'],
        y=monthly_revenue['total_revenue'],
        name='Revenue',
        marker_color='royalblue',
        hovertemplate='Month: %{x}<br>Revenue: BRL %{y:,.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Chart 2 — Order Volume Line
fig.add_trace(
    go.Scatter(
        x=monthly_revenue['month'],
        y=monthly_revenue['total_orders'],
        name='Orders',
        mode='lines+markers',
        line=dict(color='orange', width=2),
        marker=dict(size=6),
        hovertemplate='Month: %{x}<br>Orders: %{y:,}<extra></extra>'
    ),
    row=1, col=2
)

# Chart 3 — Average Order Value
fig.add_trace(
    go.Scatter(
        x=monthly_revenue['month'],
        y=monthly_revenue['avg_order_value'],
        name='Avg Order Value',
        mode='lines+markers',
        line=dict(color='green', width=2),
        fill='tozeroy',
        fillcolor='rgba(0,255,0,0.1)',
        hovertemplate='Month: %{x}<br>Avg Value: BRL %{y:,.2f}<extra></extra>'
    ),
    row=2, col=1
)

# Chart 4 — Revenue Growth Rate
colors = ['red' if x < 0 else 'green' 
          for x in monthly_revenue['revenue_growth'].fillna(0)]

fig.add_trace(
    go.Bar(
        x=monthly_revenue['month'],
        y=monthly_revenue['revenue_growth'],
        name='Growth Rate',
        marker_color=colors,
        hovertemplate='Month: %{x}<br>Growth: %{y:.1f}%<extra></extra>'
    ),
    row=2, col=2
)

# Update layout
fig.update_layout(
    title=dict(
        text='🛒 Olist E-Commerce Analytics Dashboard',
        font=dict(size=20),
        x=0.5
    ),
    height=700,
    showlegend=False,
    template='plotly_dark',
    paper_bgcolor='#1e1e2e',
    plot_bgcolor='#1e1e2e',
    font=dict(color='white')
)

# Update all x-axes
fig.update_xaxes(tickangle=45)

# Save as interactive HTML
output_path = os.path.expanduser('~/Documents/Ecommers Project/dashboard/revenue_dashboard.html')
fig.write_html(output_path)

# Open directly in browser automatically
import webbrowser
webbrowser.open('file://' + output_path)
print("Interactive dashboard saved and opened in browser!")

Interactive dashboard saved and opened in browser!


In [21]:
#Top Product Categories Analysis
full_data = (order_items
    .merge(orders, on='order_id')
    .merge(order_payments, on='order_id')
    .merge(products, on='product_id')
    .merge(category_translation, on='product_category_name', how='left')
    .merge(order_reviews, on='order_id', how='left'))

full_delivered = full_data[full_data['order_status'] == 'delivered'].copy()

# Drop period columns and convert datetime
full_delivered = full_delivered.drop(columns=['order_month', 'order_year'], errors='ignore')
for col in full_delivered.select_dtypes(include=['datetime64[ns]']).columns:
    full_delivered[col] = full_delivered[col].astype(str)

query2 = """
SELECT 
    product_category_name_english AS category,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM full_delivered
WHERE product_category_name_english IS NOT NULL
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10
"""

top_categories = ps.sqldf(query2, locals())
print("=== TOP 10 CATEGORIES BY REVENUE ===")
print(top_categories.to_string())

=== TOP 10 CATEGORIES BY REVENUE ===
                category  total_orders  total_revenue  avg_review_score
0         bed_bath_table          9272     1723932.14              3.92
1          health_beauty          8646     1625923.50              4.19
2  computers_accessories          6530     1563315.62              3.99
3        furniture_decor          6307     1408110.04              3.96
4          watches_gifts          5495     1388699.25              4.07
5         sports_leisure          7530     1357249.46              4.16
6             housewares          5743     1072820.85              4.12
7                   auto          3810      835782.91              4.11
8           garden_tools          3448      813055.77              4.08
9             cool_stuff          3559      746763.39              4.19


In [22]:
# Interactive Category Dashboard
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        '💰 Top 10 Categories by Revenue',
        '⭐ Average Review Score by Category'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Chart 1 — Revenue by Category
fig2.add_trace(
    go.Bar(
        x=top_categories['total_revenue'],
        y=top_categories['category'],
        orientation='h',
        name='Revenue',
        marker=dict(
            color=top_categories['total_revenue'],
            colorscale='Blues',
            showscale=True
        ),
        hovertemplate='Category: %{y}<br>Revenue: BRL %{x:,.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Chart 2 — Review Score by Category
colors_review = ['red' if x < 4.0 else 'green' 
                 for x in top_categories['avg_review_score']]

fig2.add_trace(
    go.Bar(
        x=top_categories['avg_review_score'],
        y=top_categories['category'],
        orientation='h',
        name='Review Score',
        marker_color=colors_review,
        hovertemplate='Category: %{y}<br>Score: %{x:.2f}<extra></extra>'
    ),
    row=1, col=2
)

# Add reference line at 4.0
fig2.add_vline(
    x=4.0, row=1, col=2,
    line_dash="dash",
    line_color="yellow",
    annotation_text="Target: 4.0",
    annotation_position="top"
)

fig2.update_layout(
    title=dict(
        text='Product Category Performance Dashboard',
        font=dict(size=20),
        x=0.5
    ),
    height=600,
    showlegend=False,
    template='plotly_dark',
    paper_bgcolor='#1e1e2e',
    plot_bgcolor='#1e1e2e',
    font=dict(color='white')
)

fig2.update_yaxes(autorange="reversed")

# Save and open in browser
output_path2 = os.path.expanduser('~/Documents/Ecommers Project/dashboard/category_dashboard.html')
fig2.write_html(output_path2)
webbrowser.open('file://' + output_path2)
print("Category dashboard saved and opened in browser!")

Category dashboard saved and opened in browser!


In [ ]:
# Customer Segmentation RFM Analysis
orders_customers = (orders
    .merge(customers, on='customer_id')
    .merge(order_payments, on='order_id'))

delivered_customers = orders_customers[orders_customers['order_status'] == 'delivered'].copy()
delivered_customers = delivered_customers.drop(columns=['order_month', 'order_year'], errors='ignore')
for col in delivered_customers.select_dtypes(include=['datetime64[ns]']).columns:
    delivered_customers[col] = delivered_customers[col].astype(str)

query3 = """
SELECT 
    customer_unique_id,
    COUNT(DISTINCT order_id) AS frequency,
    ROUND(SUM(payment_value), 2) AS monetary,
    ROUND(AVG(payment_value), 2) AS avg_order_value
FROM delivered_customers
GROUP BY customer_unique_id
ORDER BY monetary DESC
"""

rfm_data = ps.sqldf(query3, locals())

# Add customer segments
def segment_customer(row):
    if row['frequency'] >= 3 and row['monetary'] >= 500:
        return 'High Value'
    elif row['frequency'] >= 2 and row['monetary'] >= 200:
        return 'Mid Value'
    elif row['monetary'] >= 100:
        return 'Regular'
    else:
        return 'Low Value'

rfm_data['segment'] = rfm_data.apply(segment_customer, axis=1)

segment_summary = rfm_data.groupby('segment').agg(
    total_customers=('customer_unique_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_order_value=('avg_order_value', 'mean')
).reset_index()

print("=== CUSTOMER SEGMENTATION ===")
print(segment_summary.to_string())
print(f"\nTotal Customers: {len(rfm_data):,}")
print(f"High Value Customers: {len(rfm_data[rfm_data['segment']=='High Value']):,}")

=== CUSTOMER SEGMENTATION ===
      segment  total_customers  total_revenue  avg_order_value
0  High Value               87       77799.62           249.44
1   Low Value            43246     2610359.26            59.11
2   Mid Value             1517      625900.20           198.19
3     Regular            48507    12108402.69           244.07

Total Customers: 93,357
High Value Customers: 87


In [26]:
#Customer Segmentation Dashboard
fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Customer Segments Distribution',
        'Revenue by Customer Segment'
    ),
    specs=[[{"type": "pie"}, {"type": "bar"}]]
)

# Chart 1 — Pie chart of customer segments
fig3.add_trace(
    go.Pie(
        labels=segment_summary['segment'],
        values=segment_summary['total_customers'],
        hole=0.4,
        marker=dict(colors=['#2196F3', '#4CAF50', '#FF9800', '#F44336']),
        hovertemplate='Segment: %{label}<br>Customers: %{value:,}<br>Percentage: %{percent}<extra></extra>'
    ),
    row=1, col=1
)

# Chart 2 — Revenue by segment
fig3.add_trace(
    go.Bar(
        x=segment_summary['segment'],
        y=segment_summary['total_revenue'],
        marker=dict(
            color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'],
        ),
        hovertemplate='Segment: %{x}<br>Revenue: BRL %{y:,.2f}<extra></extra>'
    ),
    row=1, col=2
)

fig3.update_layout(
    title=dict(
        text='Customer Segmentation Analysis — RFM Model',
        font=dict(size=20),
        x=0.5
    ),
    height=500,
    showlegend=True,
    template='plotly_dark',
    paper_bgcolor='#1e1e2e',
    plot_bgcolor='#1e1e2e',
    font=dict(color='white')
)

output_path3 = os.path.expanduser('~/Documents/Ecommers Project/dashboard/customer_segmentation.html')
fig3.write_html(output_path3)
webbrowser.open('file://' + output_path3)
print("Customer segmentation dashboard saved and opened!")

Customer segmentation dashboard saved and opened!


In [27]:
#  Delivery Performance Analysis
delivery_data = orders[orders['order_status'] == 'delivered'].copy()
delivery_data = delivery_data.dropna(subset=['order_delivered_customer_date'])

# Calculate delivery days
delivery_data['delivery_days'] = (
    delivery_data['order_delivered_customer_date'] - 
    delivery_data['order_purchase_timestamp']
).dt.days

# On time vs late
delivery_data['on_time'] = (
    delivery_data['order_delivered_customer_date'] <= 
    delivery_data['order_estimated_delivery_date']
)

# Convert for SQL
delivery_sql = delivery_data.copy()
delivery_sql = delivery_sql.drop(columns=['order_month', 'order_year'], errors='ignore')
for col in delivery_sql.select_dtypes(include=['datetime64[ns]']).columns:
    delivery_sql[col] = delivery_sql[col].astype(str)

query4 = """
SELECT
    substr(order_purchase_timestamp, 1, 7) AS month,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN on_time = 1 THEN 1 ELSE 0 END) AS on_time_orders,
    ROUND(100.0 * SUM(CASE WHEN on_time = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS on_time_rate,
    ROUND(AVG(delivery_days), 1) AS avg_delivery_days
FROM delivery_sql
GROUP BY month
ORDER BY month
"""

delivery_perf = ps.sqldf(query4, locals())

print("=== DELIVERY PERFORMANCE ===")
print(delivery_perf.to_string())
print(f"\nOverall On-Time Rate: {delivery_data['on_time'].mean()*100:.1f}%")
print(f"Average Delivery Days: {delivery_data['delivery_days'].mean():.1f} days")
print(f"Fastest Delivery: {delivery_data['delivery_days'].min()} days")
print(f"Slowest Delivery: {delivery_data['delivery_days'].max()} days")

=== DELIVERY PERFORMANCE ===
      month  total_orders  on_time_orders  on_time_rate  avg_delivery_days
0   2016-09             1               0          0.00              54.00
1   2016-10           265             262         98.90              19.10
2   2016-12             1               1        100.00               4.00
3   2017-01           750             727         96.90              12.10
4   2017-02          1653            1600         96.80              12.60
5   2017-03          2546            2404         94.40              12.40
6   2017-04          2303            2122         92.10              14.40
7   2017-05          3545            3417         96.40              10.80
8   2017-06          3135            3014         96.10              11.50
9   2017-07          3872            3739         96.60              11.10
10  2017-08          4193            4054         96.70              10.70
11  2017-09          4150            3934         94.80              11

In [28]:
# Delivery Performance Dashboard
fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'On-Time Delivery Rate by Month (%)',
        'Average Delivery Days by Month',
        'Delivery Days Distribution',
        'On-Time vs Late Orders'
    ),
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "pie"}]]
)

# Chart 1 — On-time rate trend
fig4.add_trace(
    go.Scatter(
        x=delivery_perf['month'],
        y=delivery_perf['on_time_rate'],
        mode='lines+markers',
        line=dict(color='#4CAF50', width=2),
        marker=dict(size=6),
        hovertemplate='Month: %{x}<br>On-Time Rate: %{y:.1f}%<extra></extra>'
    ),
    row=1, col=1
)

# Add 90% target line
fig4.add_hline(
    y=90, row=1, col=1,
    line_dash="dash",
    line_color="yellow",
    annotation_text="Target: 90%"
)

# Chart 2 — Avg delivery days
fig4.add_trace(
    go.Bar(
        x=delivery_perf['month'],
        y=delivery_perf['avg_delivery_days'],
        marker=dict(
            color=delivery_perf['avg_delivery_days'],
            colorscale='RdYlGn_r'
        ),
        hovertemplate='Month: %{x}<br>Avg Days: %{y:.1f}<extra></extra>'
    ),
    row=1, col=2
)

# Chart 3 — Distribution of delivery days
fig4.add_trace(
    go.Histogram(
        x=delivery_data['delivery_days'].clip(0, 50),
        nbinsx=50,
        marker_color='royalblue',
        hovertemplate='Days: %{x}<br>Count: %{y}<extra></extra>'
    ),
    row=2, col=1
)

# Chart 4 — On-time vs Late pie
on_time_count = delivery_data['on_time'].sum()
late_count = len(delivery_data) - on_time_count

fig4.add_trace(
    go.Pie(
        labels=['On Time', 'Late'],
        values=[on_time_count, late_count],
        hole=0.4,
        marker=dict(colors=['#4CAF50', '#F44336']),
        hovertemplate='%{label}: %{value:,} orders (%{percent})<extra></extra>'
    ),
    row=2, col=2
)

fig4.update_layout(
    title=dict(
        text='Delivery Performance Analytics Dashboard',
        font=dict(size=20),
        x=0.5
    ),
    height=700,
    showlegend=False,
    template='plotly_dark',
    paper_bgcolor='#1e1e2e',
    plot_bgcolor='#1e1e2e',
    font=dict(color='white')
)

fig4.update_xaxes(tickangle=45)

output_path4 = os.path.expanduser('~/Documents/Ecommers Project/dashboard/delivery_dashboard.html')
fig4.write_html(output_path4)
webbrowser.open('file://' + output_path4)
print("Delivery dashboard saved and opened!")

Delivery dashboard saved and opened!


In [ ]:
# Master KPI Summary Dashboard
fig5 = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        'Total Revenue by Year',
        'Orders by State (Top 10)',
        'Payment Methods',
        'Review Score Distribution',
        'Order Status Breakdown',
        'Seller Performance',
        'Monthly Revenue Trend',
        'Delivery vs Revenue Correlation',
        'Customer Value Distribution'
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}, {"type": "pie"}],
        [{"type": "histogram"}, {"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "scatter"}, {"type": "histogram"}]
    ]
)

# Prepare data
# Revenue by year
year_revenue = orders.merge(order_payments, on='order_id')
year_revenue = year_revenue[year_revenue['order_status'] == 'delivered']
year_revenue['year'] = year_revenue['order_purchase_timestamp'].dt.year
year_rev = year_revenue.groupby('year')['payment_value'].sum().reset_index()

# Orders by state
state_orders = orders.merge(customers, on='customer_id')
state_orders = state_orders[state_orders['order_status'] == 'delivered']
top_states = state_orders.groupby('customer_state').size().reset_index(name='orders').nlargest(10, 'orders')

# Payment methods
payment_methods = order_payments.groupby('payment_type')['payment_value'].sum().reset_index()

# Review scores
review_dist = order_reviews['review_score'].value_counts().reset_index()
review_dist.columns = ['score', 'count']
review_dist = review_dist.sort_values('score')

# Order status — clean small categories
order_status = orders['order_status'].value_counts().reset_index()
order_status.columns = ['status', 'count']
order_status_clean = order_status.copy()
order_status_clean['status'] = order_status_clean.apply(
    lambda x: x['status'] if x['count'] > 500 else 'other', axis=1
)
order_status_clean = order_status_clean.groupby('status')['count'].sum().reset_index()
order_status_clean = order_status_clean.sort_values('count', ascending=False)

# Seller performance
seller_perf = (order_items
    .merge(order_payments, on='order_id')
    .groupby('seller_id')['payment_value']
    .sum()
    .reset_index()
    .nlargest(10, 'payment_value'))

# Chart 1 — Revenue by Year
fig5.add_trace(
    go.Bar(
        x=year_rev['year'].astype(str),
        y=year_rev['payment_value'],
        marker_color=['#2196F3', '#4CAF50', '#FF9800'],
        hovertemplate='Year: %{x}<br>Revenue: BRL %{y:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Chart 2 — Top States
fig5.add_trace(
    go.Bar(
        x=top_states['orders'],
        y=top_states['customer_state'],
        orientation='h',
        marker_color='royalblue',
        hovertemplate='State: %{y}<br>Orders: %{x:,}<extra></extra>'
    ),
    row=1, col=2
)

# Chart 3 — Payment Methods Pie
fig5.add_trace(
    go.Pie(
        labels=payment_methods['payment_type'],
        values=payment_methods['payment_value'],
        hole=0.4,
        textposition='outside',
        textinfo='label+percent',
        marker=dict(colors=['#2196F3', '#4CAF50', '#FF9800', '#F44336']),
        hovertemplate='%{label}: BRL %{value:,.0f} (%{percent})<extra></extra>'
    ),
    row=1, col=3
)

# Chart 4 — Review Score Distribution
fig5.add_trace(
    go.Bar(
        x=review_dist['score'].astype(str),
        y=review_dist['count'],
        marker=dict(
            color=['#F44336', '#FF5722', '#FF9800', '#8BC34A', '#4CAF50']
        ),
        hovertemplate='Score: %{x}<br>Count: %{y:,}<extra></extra>'
    ),
    row=2, col=1
)

# Chart 5 — Order Status Bar (cleaner than pie)
fig5.add_trace(
    go.Bar(
        x=order_status_clean['status'],
        y=order_status_clean['count'],
        marker=dict(
            color=['#2196F3', '#4CAF50', '#FF9800', '#F44336',
                   '#9C27B0', '#00BCD4', '#FF5722']
        ),
        hovertemplate='Status: %{x}<br>Count: %{y:,}<extra></extra>'
    ),
    row=2, col=2
)

# Chart 6 — Top Sellers
fig5.add_trace(
    go.Bar(
        x=seller_perf['payment_value'],
        y=seller_perf['seller_id'].str[:8],
        orientation='h',
        marker_color='purple',
        hovertemplate='Seller: %{y}<br>Revenue: BRL %{x:,.0f}<extra></extra>'
    ),
    row=2, col=3
)

# Chart 7 — Monthly Revenue
fig5.add_trace(
    go.Scatter(
        x=monthly_revenue['month'],
        y=monthly_revenue['total_revenue'],
        mode='lines',
        line=dict(color='#4CAF50', width=2),
        fill='tozeroy',
        fillcolor='rgba(76,175,80,0.2)',
        hovertemplate='Month: %{x}<br>Revenue: BRL %{y:,.0f}<extra></extra>'
    ),
    row=3, col=1
)

# Chart 8 — Delivery vs Revenue Correlation
monthly_merged = monthly_revenue.merge(
    delivery_perf[['month', 'on_time_rate', 'avg_delivery_days']],
    on='month',
    how='inner'
)

fig5.add_trace(
    go.Scatter(
        x=monthly_merged['avg_delivery_days'],
        y=monthly_merged['total_revenue'],
        mode='markers',
        marker=dict(
            size=10,
            color=monthly_merged['on_time_rate'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title='On-Time %')
        ),
        hovertemplate='Delivery Days: %{x}<br>Revenue: BRL %{y:,.0f}<extra></extra>'
    ),
    row=3, col=2
)

# Chart 9 — Customer Value Distribution
fig5.add_trace(
    go.Histogram(
        x=rfm_data['monetary'].clip(0, 1000),
        nbinsx=50,
        marker_color='orange',
        hovertemplate='Value: BRL %{x}<br>Customers: %{y:,}<extra></extra>'
    ),
    row=3, col=3
)

# Update layout
fig5.update_layout(
    title=dict(
        text='Olist E-Commerce — Master Analytics Dashboard',
        font=dict(size=22, color='white'),
        x=0.5
    ),
    height=1200,
    showlegend=False,
    template='plotly_dark',
    paper_bgcolor='#1e1e2e',
    plot_bgcolor='#1e1e2e',
    font=dict(color='white')
)

fig5.update_xaxes(tickangle=45)
fig5.update_yaxes(autorange="reversed", row=1, col=2)
fig5.update_yaxes(autorange="reversed", row=2, col=3)

# Save and open
output_path5 = os.path.expanduser('~/Documents/Ecommers Project/dashboard/master_dashboard.html')
fig5.write_html(output_path5)
webbrowser.open('file://' + output_path5)
print("Master dashboard saved and opened!")

Master dashboard saved and opened!


In [35]:
#  Professional KPI Card Dashboard
fig_kpi = go.Figure()

# Calculate KPIs
total_revenue = order_payments['payment_value'].sum()
total_orders = orders['order_id'].nunique()
avg_order_value = order_payments['payment_value'].mean()
on_time_rate = delivery_data['on_time'].mean() * 100
avg_review = order_reviews['review_score'].mean()
total_customers = customers['customer_unique_id'].nunique()
total_sellers = sellers['seller_id'].nunique()
delivered_orders = len(orders[orders['order_status'] == 'delivered'])
delivery_rate = delivered_orders / total_orders * 100

# Build HTML dashboard
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Olist E-Commerce KPI Dashboard</title>
    <meta charset="UTF-8">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{
            background: #0f0f1a;
            font-family: 'Segoe UI', sans-serif;
            color: white;
            padding: 20px;
        }}
        .header {{
            text-align: center;
            padding: 20px 0;
            border-bottom: 1px solid #333;
            margin-bottom: 25px;
        }}
        .header h1 {{
            font-size: 28px;
            color: #4CAF50;
            letter-spacing: 2px;
        }}
        .header p {{
            color: #888;
            font-size: 14px;
            margin-top: 5px;
        }}
        .kpi-grid {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 15px;
            margin-bottom: 25px;
        }}
        .kpi-card {{
            background: #1e1e2e;
            border-radius: 12px;
            padding: 20px;
            border: 1px solid #333;
            transition: transform 0.2s;
        }}
        .kpi-card:hover {{
            transform: translateY(-3px);
            border-color: #4CAF50;
        }}
        .kpi-label {{
            font-size: 12px;
            color: #888;
            text-transform: uppercase;
            letter-spacing: 1px;
            margin-bottom: 8px;
        }}
        .kpi-value {{
            font-size: 32px;
            font-weight: bold;
            color: white;
        }}
        .kpi-sub {{
            font-size: 12px;
            margin-top: 5px;
        }}
        .positive {{ color: #4CAF50; }}
        .negative {{ color: #F44336; }}
        .neutral {{ color: #2196F3; }}
        .kpi-icon {{
            font-size: 24px;
            float: right;
            margin-top: -5px;
        }}
        .charts-grid {{
            display: grid;
            grid-template-columns: 2fr 1fr;
            gap: 15px;
            margin-bottom: 25px;
        }}
        .chart-card {{
            background: #1e1e2e;
            border-radius: 12px;
            padding: 20px;
            border: 1px solid #333;
        }}
        .chart-title {{
            font-size: 14px;
            color: #aaa;
            margin-bottom: 15px;
            text-transform: uppercase;
            letter-spacing: 1px;
        }}
        .bottom-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr 1fr;
            gap: 15px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
        }}
        th {{
            color: #888;
            text-align: left;
            padding: 8px 0;
            border-bottom: 1px solid #333;
            font-weight: normal;
            font-size: 11px;
            text-transform: uppercase;
        }}
        td {{
            padding: 8px 0;
            border-bottom: 1px solid #222;
            color: #ddd;
        }}
        td:last-child {{ text-align: right; color: #4CAF50; }}
        .gauge-container {{
            display: flex;
            justify-content: center;
            align-items: center;
            height: 200px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🛒 OLIST E-COMMERCE ANALYTICS</h1>
        <p>Brazilian E-Commerce Performance Dashboard | 2016 - 2018 | Built with Python & Plotly</p>
    </div>

    <!-- KPI Cards Row -->
    <div class="kpi-grid">
        <div class="kpi-card">
            <div class="kpi-icon">💰</div>
            <div class="kpi-label">Total Revenue</div>
            <div class="kpi-value">BRL {total_revenue/1e6:.1f}M</div>
            <div class="kpi-sub positive">▲ 127% YoY Growth</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">📦</div>
            <div class="kpi-label">Total Orders</div>
            <div class="kpi-value">{total_orders:,}</div>
            <div class="kpi-sub positive">▲ 98% Delivered</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">💳</div>
            <div class="kpi-label">Avg Order Value</div>
            <div class="kpi-value">BRL {avg_order_value:.0f}</div>
            <div class="kpi-sub neutral">Stable across 2018</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">⭐</div>
            <div class="kpi-label">Avg Review Score</div>
            <div class="kpi-value">{avg_review:.1f}/5</div>
            <div class="kpi-sub positive">▲ Strong satisfaction</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">🚚</div>
            <div class="kpi-label">On-Time Delivery</div>
            <div class="kpi-value">{on_time_rate:.1f}%</div>
            <div class="kpi-sub positive">▲ Above 90% target</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">👥</div>
            <div class="kpi-label">Total Customers</div>
            <div class="kpi-value">{total_customers:,}</div>
            <div class="kpi-sub neutral">Unique buyers</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">🏪</div>
            <div class="kpi-label">Active Sellers</div>
            <div class="kpi-value">{total_sellers:,}</div>
            <div class="kpi-sub neutral">Marketplace partners</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-icon">📅</div>
            <div class="kpi-label">Avg Delivery Days</div>
            <div class="kpi-value">{delivery_data['delivery_days'].mean():.1f}</div>
            <div class="kpi-sub positive">▲ Improved over time</div>
        </div>
    </div>

    <!-- Charts Row -->
    <div class="charts-grid">
        <div class="chart-card">
            <div class="chart-title">📈 Monthly Revenue Trend</div>
            <div id="revenueChart"></div>
        </div>
        <div class="chart-card">
            <div class="chart-title">💳 Payment Methods</div>
            <div id="paymentChart"></div>
        </div>
    </div>

    <!-- Bottom Row -->
    <div class="bottom-grid">
        <div class="chart-card">
            <div class="chart-title">🏆 Top Categories by Revenue</div>
            <table>
                <tr>
                    <th>#</th>
                    <th>Category</th>
                    <th>Revenue</th>
                </tr>
                {chr(10).join([f'<tr><td>{i+1}</td><td>{row["category"]}</td><td>BRL {row["total_revenue"]/1000:.0f}K</td></tr>' 
                              for i, row in top_categories.head(8).iterrows()])}
            </table>
        </div>
        <div class="chart-card">
            <div class="chart-title">⭐ Review Distribution</div>
            <div id="reviewChart"></div>
        </div>
        <div class="chart-card">
            <div class="chart-title">🗺️ Top States by Orders</div>
            <table>
                <tr>
                    <th>#</th>
                    <th>State</th>
                    <th>Orders</th>
                </tr>
                {chr(10).join([f'<tr><td>{i+1}</td><td>{row["customer_state"]}</td><td>{row["orders"]:,}</td></tr>' 
                              for i, row in top_states.head(8).iterrows()])}
            </table>
        </div>
    </div>

    <script>
        // Revenue Chart
        var revenueData = {{
            x: {list(monthly_revenue['month'])},
            y: {list(monthly_revenue['total_revenue'])},
            type: 'scatter',
            mode: 'lines',
            fill: 'tozeroy',
            line: {{color: '#4CAF50', width: 2}},
            fillcolor: 'rgba(76,175,80,0.15)'
        }};
        Plotly.newPlot('revenueChart', [revenueData], {{
            paper_bgcolor: 'transparent',
            plot_bgcolor: 'transparent',
            font: {{color: 'white', size: 11}},
            margin: {{t:10, r:10, b:40, l:60}},
            height: 220,
            xaxis: {{gridcolor: '#333', tickangle: 45}},
            yaxis: {{gridcolor: '#333'}}
        }});

        // Payment Chart
        var paymentData = {{
            labels: {list(payment_methods['payment_type'])},
            values: {list(payment_methods['payment_value'])},
            type: 'pie',
            hole: 0.5,
            marker: {{colors: ['#2196F3', '#4CAF50', '#FF9800', '#F44336']}},
            textinfo: 'percent',
            textposition: 'inside',
            hoverinfo: 'label+percent+value',
        }};
        Plotly.newPlot('paymentChart', [paymentData], {{
            paper_bgcolor: 'transparent',
            plot_bgcolor: 'transparent',
            font: {{color: 'white', size: 11}},
            margin: {{t:10, r:10, b:10, l:10}},
            height: 220,
            showlegend: false
        }});

        // Review Chart
        var reviewData = {{
            x: {list(review_dist['score'].astype(str))},
            y: {list(review_dist['count'])},
            type: 'bar',
            marker: {{color: ['#F44336','#FF5722','#FF9800','#8BC34A','#4CAF50']}}
        }};
        Plotly.newPlot('reviewChart', [reviewData], {{
            paper_bgcolor: 'transparent',
            plot_bgcolor: 'transparent',
            font: {{color: 'white', size: 11}},
            margin: {{t:10, r:10, b:40, l:50}},
            height: 220,
            xaxis: {{gridcolor: '#333'}},
            yaxis: {{gridcolor: '#333'}}
        }});
    </script>
</body>
</html>
"""

# Save and open
output_kpi = os.path.expanduser('~/Documents/Ecommers Project/dashboard/kpi_dashboard.html')
with open(output_kpi, 'w') as f:
    f.write(html_content)

webbrowser.open('file://' + output_kpi)
print("KPI Dashboard saved and opened!")

KPI Dashboard saved and opened!
